In [1]:
import math
import os
import time
from functools import reduce
from typing import List

import torch
import torchinfo
from torch import nn
from torch.nn import functional as F
from tqdm.auto import trange
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
from torch.nn.attention.flex_attention import create_block_mask, flex_attention
torch._inductor.config.realize_opcount_threshold = 500


configs = [[128, 64], [256, 32], [512, 16]]



class Permute(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        return x.permute(0,2,1)
block_masks = {}

for config in configs:
    d_model, patch = config
    def headlocal(b, h, q_idx, kv_idx):
        x = q_idx//(patch*patch); idx2 = q_idx-x*patch*patch
        y = idx2//patch; z = q_idx%patch
        x1 = kv_idx//(patch*patch); idx2 = kv_idx-x1*patch*patch
        y1 = idx2//patch; z1 = idx2%patch
        mask = ((x-x1).abs() <= 2) & ((y-y1).abs() <= 2) & ((z-z1).abs() <= 2)
#        mask = ((x-x1).abs() <= (h*2+1)) & ((y-y1).abs() <= (h*2+1)) & ((z-z1).abs() <= (h*2+1))
        return mask
    torch.cuda.empty_cache()
    block_mask = create_block_mask(headlocal, B=None, H=None, Q_LEN=patch**3, KV_LEN=patch**3,_compile=True)
    block_masks[str(patch)] = block_mask

from torch.nn.attention.flex_attention import flex_attention
flex_attention = torch.compile(flex_attention,dynamic=False)

class Flextension(nn.Module):
    def __init__(self,in_proj_weight,in_proj_bias,out_proj_weight,out_proj_bias,block_mask,attn_heads=4):
        super().__init__()
        self.batch_first = True
        self._qkv_same_embed_dim = True
        self.in_proj_weight = in_proj_weight
        self.in_proj_bias = in_proj_bias
        self.out_proj_weight = out_proj_weight
        self.out_proj_bias = out_proj_bias
        self.attn_heads = attn_heads
        self.num_heads = attn_heads
        self.kernel_options = {"BLOCK_M": 32,"BLOCK_N": 32,"BLOCK_M1": 16,"BLOCK_N1": 32,"BLOCK_M2": 32,"BLOCK_N2": 16,}
        #self.kernel_options = {"BLOCK_M": 64,"BLOCK_N": 64,"BLOCK_M1": 32,"BLOCK_N1": 64,"BLOCK_M2": 64,"BLOCK_N2": 32,}
        self.block_mask = block_mask#create_block_mask(poolS, B=None, H=None, Q_LEN=S, KV_LEN=S)###.to('cuda')

    def forward(self,x,x1,x2,attn_mask=None,key_padding_mask=None,need_weights=False,is_causal=False):
        q,k,v = F.linear(x,self.in_proj_weight,bias=self.in_proj_bias).chunk(3,-1)
        q_ = q.unflatten(-1,(self.attn_heads,-1)).transpose(2,1)
        k_ = k.unflatten(-1,(self.attn_heads,-1)).transpose(2,1)
        v_ = v.unflatten(-1,(self.attn_heads,-1)).transpose(2,1)
        y_ = flex_attention(q_, k_, v_, kernel_options=self.kernel_options, block_mask=self.block_mask)
        y = y_.transpose(2,1).flatten(2,-1)
        #y = F.scaled_dot_product_attention(q_,k_,v_).transpose(2,1).flatten(2,-1)
        y = F.linear(y,self.out_proj_weight,bias=self.out_proj_bias)
        #y = self.self_attn(q,k,v)[0]
        return y
   


class FlexFormer(nn.TransformerEncoderLayer):
    def __init__(self,d_model=64, nhead=4, dim_feedforward=256, dropout=0.0, activation='gelu', layer_norm_eps=1e-5, batch_first=True, mask=None):
        super().__init__(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation=activation, layer_norm_eps=layer_norm_eps, batch_first=batch_first)
        self.block_mask = mask
        self.self_attn = Flextension(self.self_attn.in_proj_weight,self.self_attn.in_proj_bias,self.self_attn.out_proj.weight,self.self_attn.out_proj.bias,self.block_mask,attn_heads=nhead)
if(False):
    x = torch.randn(1,128,64,64,64).cuda()
    for _ in range(2):
        with torch.no_grad():
            with torch.amp.autocast('cuda',dtype=torch.bfloat16):
                y = flex0(x)
                torch.cuda.synchronize()
    for _ in trange(20):
        with torch.no_grad():
            with torch.amp.autocast('cuda',dtype=torch.bfloat16):
                y = flex0(x)
                torch.cuda.synchronize()
    print(y.shape)    
#shape 128,64,64,64
#Conv 5x5x5 23.69 it/s, 16.4M params
#FullFormer 0.76 it/s, 0.83M params
#FlexFormer 21.33 it/s, 0.83M params



/home/jupyter-mattiastest/pt25nnunet/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
for config in configs:
    print('config:', config)
    model_dim,patch = config
    block_mask_config = block_masks[str(patch)]
#    x = torch.randn(1, config[1]^3, config[0]).cuda()
    x = torch.randn(1, config[0], config[1], config[1], config[1]).cuda()

    flex4 = nn.Sequential(nn.Flatten(2,-1),Permute(),*[FlexFormer(d_model=model_dim,nhead=4, dim_feedforward=model_dim*4, dropout=0.0,\
                     activation='gelu', layer_norm_eps=1e-5, batch_first=True, mask=block_mask_config).cuda() for _ in range(4)],Permute(),nn.Unflatten(-1,(patch,patch,patch))).cuda()

    for _ in range(2):
        with torch.no_grad():
            with torch.amp.autocast('cuda',dtype=torch.bfloat16):
                y = flex4(x)
                torch.cuda.synchronize()
    for _ in trange(20):
        with torch.no_grad():
            with torch.amp.autocast('cuda',dtype=torch.bfloat16):
                y = flex4(x)
                torch.cuda.synchronize()

#    x = torch.randn(1, config[1] ^ 3, config[0]).cuda()
    print(x.shape)
    

config: [128, 64]


100%|██████████| 20/20 [00:01<00:00, 12.96it/s]


torch.Size([1, 128, 64, 64, 64])
config: [256, 32]


100%|██████████| 20/20 [00:00<00:00, 79.22it/s]


torch.Size([1, 256, 32, 32, 32])
config: [512, 16]


100%|██████████| 20/20 [00:00<00:00, 250.36it/s]

torch.Size([1, 512, 16, 16, 16])


In [2]:
print('locatn5')
print('-----')
for config in configs[:1]:
    print('config:', config)

    model = FlexConv(attention_mode='local', in_channels=config[0], out_channels=config[0],
                     patch_size=[config[1], config[1], config[1]], n_heads=1, kernel=5, n_layer=4).cuda()
    #model = torch.compile(model, dynamic=False)
    x = torch.randn(1, config[1] ^ 3, config[0]).cuda()
    y = model(x)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(1):
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            y = model(x)
            y.sum().backward()
    torch.cuda.synchronize()
    t1 = time.time()

    print('# params [M]:', torchinfo.summary(model).total_params * 1e-6)
    print('runtime [ms]:', ((t1 - t0) / 20) * 1e3)
    print('memory [GB]:', torch.cuda.max_memory_allocated(0) / 1024 ** 3)
    print('--')

locatn5
-----
config: [128, 64]


BackendCompilerFailed: backend='inductor' raised:
LoweringException: AssertionError: 
  target: flex_attention
  args[0]: TensorBox(StorageBox(
    InputBuffer(name='primals_1', layout=FixedLayout('cuda', torch.float32, size=[1, 1, 67, 128], stride=[25728, 128, 384, 1]))
  ))
  args[1]: TensorBox(StorageBox(
    InputBuffer(name='primals_2', layout=FixedLayout('cuda', torch.float32, size=[1, 1, 67, 128], stride=[25728, 128, 384, 1]))
  ))
  args[2]: TensorBox(StorageBox(
    InputBuffer(name='primals_3', layout=FixedLayout('cuda', torch.float32, size=[1, 1, 67, 128], stride=[25728, 128, 384, 1]))
  ))
  args[3]: Subgraph(name='sdpa_score0', graph_module=<lambda>(), graph=None)
  args[4]: (TensorBox(StorageBox(
    InputBuffer(name='primals_4', layout=FixedLayout('cuda', torch.int32, size=[1, 1, 2048], stride=[2048, 2048, 1]))
  )), TensorBox(StorageBox(
    InputBuffer(name='primals_5', layout=FixedLayout('cuda', torch.int32, size=[1, 1, 2048, 2048], stride=[4194304, 4194304, 2048, 1])

In [ ]:
# print()
# print('##################')
# print()
#
# print('atnpyr')
# print('-----')
# for config in configs:
#     print('config:', config)
#     try:
#         model = FlexConv(attention_mode='local', in_channels=config[0], out_channels=config[0],
#                          patch_size=[config[1], config[1], config[1]], n_heads=4, kernel=3, n_layer=4).cuda()
#         model = torch.compile(model)
#         x = torch.randn(1, config[0], config[1], config[1], config[1]).cuda()
#         y = model(x)
#         torch.cuda.empty_cache();
#         torch.cuda.reset_peak_memory_stats()
#         torch.cuda.synchronize()
#         t0 = time.time()
#         for _ in range(20):
#             with torch.amp.autocast('cuda', dtype=torch.bfloat16):
#                 y = model(x)
#                 y.sum().backward()
#         torch.cuda.synchronize()
#         t1 = time.time()
#
#         print('# params [M]:', torchinfo.summary(model).total_params * 1e-6)
#         print('runtime [ms]:', ((t1 - t0) / 20) * 1e3)
#         print('memory [GB]:', torch.cuda.max_memory_allocated(0) / 1024 ** 3)
#         print('--')
#     except Exception as error:
#         print(type(error).__name__)
#
# print()
# print('##################')
# print()
